# A Simple Tutorial for TIC

## Installation

```bash
git clone https://github.com/cellethology/tic.git
cd tic
pip install -e .
```

## Load single cell data with anndata
To run tic pipeline, you have to make sure your anndata object has the following fields(minimum requirement):
- `X`: gene expression matrix
- `var`: gene names
- `obsm`: 'spatial' (x,y coordinates)

Optional:
- `obs`: cell type annotation
    please store cell type annotation in `adata.obs['cell_type']`, key = 'cell_type' (This will be used in feature extraction: select centre types)

Here is an example of how to load the CODEX UPMC data:
this will automatically download the data from the source and load it into anndata object.
The default data is stored in user's home directory(~). at .cache/tic/

If you want to remove the cache, you can run:
```bash
rm -rf ~/.cache/tic
```
or
```python
from tic.data import list_datasets, remove_cache, remove_dataset

list_datasets() # this will list all the datasets in the cache
remove_cache() # this will remove all the data in the cache
# if you want to remove a specific dataset, you can run:
remove_dataset("codex_upmc")
```







In [ ]:
from tic.data import load_codex_upmc
example_data = load_codex_upmc()
print(example_data)
print(example_data.obs['cell_type'].unique())

## Extract features from subgraph-based feature extractors
We have wrapped the subgraph-based feature extractors in the `tic.wrappers` module.
The following is an example of how to use the `FeatureWrapper` to extract features from the CODEX UPMC data.

Allowed subgraph strategies:
["knn", "radius", "k_hop_shortest", "k_hop_pyg", "bfs_python", "slice_adj"]

for 'k_hop_pyg' strategy, you need to install pytorch and torch-geometric(based on: torch_geometric.k_hop_subgraph)
```bash
pip install torch torch-geometric
```



In [ ]:
from tic.wrappers import FeatureWrapper
print(FeatureWrapper.recipes())

### if you want to use radius strategy, you need to specify the radius
Here we provide a function to estimate the radius based on the target microenv_size(default=30)
```python
from tic.graph.utils import estimate_radius
estimate_radius(example_data)
```



In [ ]:
from tic.graph.utils import estimate_radius
estimate_radius(example_data)

In [ ]:
from tic.wrappers import FeatureWrapper
feature_wrapper = FeatureWrapper(
    recipe="tme_default", 
    centre_types=["Tumor"], # This is compulsory, you must specify the centre type
    graph_params={'method':'voronoi'}, # You need to specify the way you want to build the image level graph
    subgraph_params={'strategy':'radius','radius':101}, # You need to specify the way you want to build the cell level subgraph
)
fea_adata = feature_wrapper.fit(example_data)
print(fea_adata)

In [ ]:
print(fea_adata.uns['X_predictors_names'])

In [ ]:
print(fea_adata.obsm['celltype_gene_count'])

## Pseudotime Inference
In order to infer pseudotime, you need to specify:

In [ ]:
from tic.wrappers import PseudotimeWrapper
pseudotime_wrapper = PseudotimeWrapper(
    rep_key="composition", # This is the key you store representation vector(in .obsm)
    output_dir='../example', # This is the directory you want to save the pseudotime result.
)
pseudotime_adata = pseudotime_wrapper.fit(fea_adata)
pseudotime_wrapper.plot(kind="pseudotime")
pseudotime_wrapper.plot(kind="cluster")
pseudotime_wrapper.metrics.head()


### Visualization: Monotonicity Metrics Bar Chart


In [ ]:
from tic.plotting import plot_monotonicity_metrics_bar
from utils.dataset.codex_upmc import UPMC_EPITHELIAL_GENES,UPMC_MESENCHYMAL_GENES
plot_monotonicity_metrics_bar(
    metrics=pseudotime_wrapper.metrics,
    show_top_n=10, # show the top n genes(include epithelial and mesenchymal genes)
    epithelial_genes = UPMC_EPITHELIAL_GENES, # will always show these genes
    mesenchymal_genes = UPMC_MESENCHYMAL_GENES, # will always show these genes
    save_path="../example/monotonicity_metrics_bar.png"
)

### Visualization: Biomarker vs Pseudotime
we provide a function to visualize the biomarker vs pseudotime



In [ ]:
from tic.plotting import plot_biomarker_trends
plot_biomarker_trends(
    adata=pseudotime_adata,
    selected_biomarkers=UPMC_EPITHELIAL_GENES + UPMC_MESENCHYMAL_GENES, # specify the biomarkers you want to plot
    x_transform="bin+normalize",
    y_transform="normalize",
    bins=100,
    save_path="../example/biomarker_vs_pseudotime.png"
)


## Causal Inference

In [ ]:

from tic.wrappers.causal import CausalWrapper

causal_wrapper = CausalWrapper(
    outcome="PanCK", # should be one of the biomarkers in adata.var_names
    feature_key="X_predictors",          # 默认即可
    include_extractors=("celltype_gene_count",),  # 只保留这一类特征
    method="granger_causality",
    bins=100,
    method_kwargs=dict(maxlag=3)
)

causal_adata = causal_wrapper.fit(pseudotime_adata)          # 结果写入 feature_adata.uns["causal_results"]
print(causal_adata)
print(causal_adata.uns['causal_results'])

In [ ]:
causal_wrapper.plot(causal_adata, kind='bar')

In [ ]:
causal_wrapper.plot(causal_adata, kind='volcano')

In [ ]:
causal_wrapper.plot(causal_adata, kind='heatmap')